## IPs and CIDR notation

**Why a support engineer needs this:** firewall rules, security groups, `ip addr`, `ip route` and allowlists all use `/24`-style notation. You need to answer **"is this IP inside that range?"** in seconds. You don't need to memorize binary to do that.

### 1. The only fact you need first

An IPv4 address is **4 numbers**, each from 0 to 255. Each number is **8 bits**, so the address is 4 × 8 = **32 bits**.
```shell
10  .  20  .  2   .  15
8 bits 8 bits 8 bits 8 bits
```
> The `/N` after an address means **"the first N bits are fixed"**. Those bits **identify the network**, and the rest vary and **identify the hosts** in it.

---

### 2. The shortcut: learn four values

Because each number is 8 bits, the common cases line up with whole numbers:

|Notation|First ___ numbers fixed|Example range|Addresses|
|---|---|---|---|
|`/8`|1 (`10`)|`10.0.0.0 – 10.255.255.255`|16,777,216|
|`/16`|2 (`10.20`)|`10.20.0.0 – 10.20.255.255`|65,536|
|`/24`|3 (`10.20.2`)|`10.20.2.0 – 10.20.2.255`|256|
|`/32`|all 4|a single address|1|

For your ticket: `10.20.2.0/24` means "the first three numbers must be `10.20.2`, and the last number can be anything". So `10.20.2.77` is inside and `10.20.1.15` isn't.

> To check whether an IP is inside a range, compare **only the fixed numbers**. The free ones don't matter.

---

### 3. The numbers that don't line up (`/20`, `/25`, ...)

You've already seen one. Your WSL address is `172.30.240.133/20`. Here the fixed part ends **in the middle** of the third number. Don't do bit math in a ticket. Use a tool, as in the Python demo below. The size rule is `addresses = 2^(32 − N)`:

|N|`/25`|`/26`|`/28`|`/20`|
|---|---|---|---|---|
|Addresses|128|64|16|4,096|

> Bigger N means a **smaller network** in size or the number of hosts it has. This catches many people: `/32` is one machine, host, and `/8` is a huge network.

---

4. Live demo (Python, nothing to install)
You know Python, so use its `ipaddress` module as a calculator.

In [3]:
import ipaddress as i

n = i.ip_network('10.20.2.0/24')
print(n.netmask, n[0], n[-1], n.num_addresses)
print(i.ip_address('10.20.1.15') in n)
print(i.ip_address('10.20.2.77') in n)
print(i.ip_address('10.20.1.15') in i.ip_network('10.20.0.0/16'))

255.255.255.0 10.20.2.0 10.20.2.255 256
False
True
True


That reproduces your ticket: `app01` (`10.20.1.15`) is outside the `/24` range and inside the `/16`.

The **netmask** `255.255.255.0` is just another way to write `/24`. The `255` octets are the fixed parts and the `0` is the free part. You'll see it in older tools and configs.

---

### 5. Read your own network
```bash
dennie@X390:~$ ip -4 addr show eth0 | grep inet
    inet 172.30.240.133/20 brd 172.30.255.255 scope global eth0
```
My WSL showed `172.30.240.133/20`. Yours will differ. Find the `/N` and say how many addresses it covers.

Then check the route table:
```bash
dennie@X390:~$ ip route
default via 172.30.240.1 dev eth0 proto kernel
172.30.240.0/20 dev eth0 proto kernel scope link src 172.30.240.133
```
Each line like `172.30.240.0/20 dev eth0 ...` is a network in CIDR form. This is the same notation used in firewall rules.

---

### 6. Try it yourself

Answer these by comparing fixed numbers first, then confirm with the Python snippet:

1. Is `192.168.5.40` inside `192.168.5.0/24`?
- ✅ Yes, `192.168.5.40` is inside `192.168.5.0/24`.
- ✅ It's inside because the **first three octets match** (`192.168.5`). The "default mask" label isn't the reason. CIDR ignores the old classes. `/24` simply means 24 fixed bits.
> 🔁 **Support rule**: compare the fixed octets, not the class.

2. Is `10.30.1.9` inside `10.20.0.0/16`?
- `10.20.0.0/16` network has fixed 2 first octets `10.20.`
But `10.30.1.9` host has different second octet `.30` meaning it doesn't match the given network ID and belongs to the different network. So, `10.30.1.9` is not inside `10.20.0.0/16`.

3. How many hosts does a `/32` rule cover, and what would you use it for?
- `/32` has a single network address, one host.
- ✅ `/32` covers exactly **one host**.
- Typical uses: allowing or blocking one specific machine, such as **"only the monitoring server `172.16.4.10/32` may reach this port"**. Also the single-address routes you've seen, like `10.255.255.254/32` on `lo`.

4. A rule says `allow from 10.20.0.0/16`. Name two subnets it covers.
-✅ `10.20.0.0/16` contains `10.20.1.0/24` and `10.20.2.0/24` because `10.20.1` and `10.20.2` both start with `10.20`.

🔁 Support rule: to compare two blocks, line up their fixed parts. One starts with the other, or they differ somewhere. No half-overlap exists.

**TICKET SCENARIO:** "Our monitoring server in `172.16.4.0/24` can't reach the new service, but the app servers in `172.16.5.0/24` can." The firewall rule allows `172.16.5.0/24`. You can see why immediately: the third number is `5`, not `4`.

**GOTCHA:** a rule that is too wide, such as `0.0.0.0/0` (everything), is a security risk. One that is too narrow locks out legitimate clients, which is what happened to `app01`.

**Course pointer (after you've practiced):** freeCodeCamp **IPv4 Logical Addressing — 48:09** and **Classful vs. Classless (CIDR) — 55:17** cover the theory behind this. Do them after the exercises above, not before.

In [ ]:
import ipaddress as i

n = i.ip_network('10.20.2.0/24')
print(n.netmask, n[0], n[-1], n.num_addresses)
print(i.ip_address('10.20.1.15') in n)
print(i.ip_address('10.20.2.77') in n)
print(i.ip_address('10.20.1.15') in i.ip_network('10.20.0.0/16'))